# Statistics of the final radiomic features

Input: the reproducible, non-redundant features in `outputs/features/final` (one CSV per filter, made by `feature_matrix.ipynb`).

**Design of the rows.** 5 patients (`case`) × 2 breasts (`side` L/R) × 4 MRI channels (`channel`) = 40 rows per filter.
The rows are *not* independent, so every test respects the pairing:

| question | test | effect size |
|---|---|---|
| do features differ between MRI channels? | Friedman (blocks = breast: case + side) | Kendall's W (0-1) |
| do features differ between left and right breast? | Wilcoxon signed-rank (pairs = case + channel) | rank-biserial r (-1..1) |
| do features differ between independent groups (e.g. a diagnosis label)? | Kruskal-Wallis / Mann-Whitney U | epsilon² / rank-biserial r |
| do all features together differ between groups? | PERMANOVA (permutations inside blocks) | R² |
| can all features together predict a group? | logistic regression, leave-one-patient-out CV, permutation test | balanced accuracy |

Non-parametric tests are used because most features are not normally distributed (see the Shapiro-Wilk summary below).
All p-values are corrected over the features with Benjamini-Hochberg (**q-values**); a feature is significant if q < `ALPHA`.

Tables are saved in `STAT_DIR`; the figures are made in `visualization.ipynb`.

In [1]:
import sys
from pathlib import Path

CODE_ROOT = Path("/Users/kamleshranabhat/Desktop/radiomics/full_pipeline")
for folder in ["", "scripts", "post_processing", "feature_analysis"]:
    if str(CODE_ROOT / folder) not in sys.path:
        sys.path.insert(0, str(CODE_ROOT / folder))

import pandas as pd
from IPython.display import display
from stat_and_viz import *

FEATURE_DIR = Path("/Users/kamleshranabhat/Desktop/radiomics/full_pipeline/outputs/features/final")
STAT_DIR = Path("/Users/kamleshranabhat/Desktop/radiomics/full_pipeline/outputs/statistics")
STAT_DIR.mkdir(parents=True, exist_ok=True)

ALPHA = 0.05            # significance level for the q-values
N_PERMUTATIONS = 999    # permutations for PERMANOVA

## 1. Load the final features

All filters merged into one table; `info` has the filter, image type and feature class of every feature.

In [2]:
table, info = load_final_features(FEATURE_DIR)
table.head()

11 filters | 40 samples | 2254 features


,case,side,channel,task,original_shape_Elongation,original_shape_Flatness,original_shape_LeastAxisLength,original_shape_MajorAxisLength,original_shape_Maximum2DDiameterColumn,original_shape_Maximum2DDiameterRow,...,lbp-3D-k_gldm_DependenceVariance,lbp-3D-k_gldm_GrayLevelNonUniformity,lbp-3D-k_gldm_LargeDependenceHighGrayLevelEmphasis,lbp-3D-k_gldm_LargeDependenceLowGrayLevelEmphasis,lbp-3D-k_gldm_SmallDependenceEmphasis,lbp-3D-k_gldm_SmallDependenceLowGrayLevelEmphasis,lbp-3D-k_ngtdm_Busyness,lbp-3D-k_ngtdm_Coarseness,lbp-3D-k_ngtdm_Contrast,lbp-3D-k_ngtdm_Strength
0,BreastMRI002,L,0,BreastMRI002_L_ch0,0.695006,0.404095,48.263987,119.437133,121.852370,153.808322,...,15.643307,48772.983009,430.459075,14.086077,0.112072,0.019689,280.446446,0.000025,0.013190,0.046948
1,BreastMRI002,L,1,BreastMRI002_L_ch1,0.695006,0.404095,48.263987,119.437133,121.852370,153.808322,...,16.488433,55567.892000,455.954793,15.680469,0.098171,0.020166,265.953658,0.000025,0.007843,0.065245
2,BreastMRI002,L,2,BreastMRI002_L_ch2,0.695006,0.404095,48.263987,119.437133,121.852370,153.808322,...,13.793798,35816.651189,411.114552,30.525976,0.100322,0.013535,225.498583,0.000028,0.018000,0.051893
3,BreastMRI002,L,3,BreastMRI002_L_ch3,0.695006,0.404095,48.263987,119.437133,121.852370,153.808322,...,24.115817,65659.044991,452.207466,31.122185,0.075317,0.015919,364.562570,0.000027,0.007621,0.046591
4,BreastMRI002,R,0,BreastMRI002_R_ch0,0.683072,0.405249,48.358348,119.329953,127.863208,153.717273,...,20.588184,61065.167955,442.039761,22.528135,0.083296,0.018150,344.180953,0.000026,0.007556,0.046557


In [3]:
info.head()

,feature,filter,image_type,feature_class,feature_name
0,original_shape_Elongation,raw_image,original,shape,Elongation
1,original_shape_Flatness,raw_image,original,shape,Flatness
2,original_shape_LeastAxisLength,raw_image,original,shape,LeastAxisLength
3,original_shape_MajorAxisLength,raw_image,original,shape,MajorAxisLength
4,original_shape_Maximum2DDiameterColumn,raw_image,original,shape,Maximum2DDiameterColumn


Number of features per filter before and after the reproducibility (ICC) and redundancy steps

In [4]:
counts = selection_counts(FEATURE_DIR)
counts.to_csv(STAT_DIR / "selection_counts.csv")
counts

,extracted,reproducible,selected,retention (%)
filter,,,,
raw_image,107,91,78,72.9
square,93,81,71,76.3
square_root,93,73,62,66.7
logarithmic,93,71,61,65.6
exponential,93,78,68,73.1
gradient,93,75,66,71.0
log,465,444,370,79.6
wavelet,744,695,589,79.2
laws,558,524,433,77.6


Number of final features per filter and feature class

In [5]:
class_counts = count_by_filter_and_class(info)
class_counts.to_csv(STAT_DIR / "features_per_filter_and_class.csv")
class_counts

feature_class,shape,firstorder,glcm,glrlm,glszm,gldm,ngtdm
filter,,,,,,,
raw_image,13,12,16,10,13,10,4
square,0,12,20,10,12,12,5
square_root,0,11,12,13,12,10,4
logarithmic,0,12,13,13,11,8,4
exponential,0,13,18,14,9,11,3
gradient,0,14,18,9,10,10,5
log,0,75,86,62,59,63,25
wavelet,0,131,134,86,104,95,39
laws,0,83,99,79,71,74,27


## 2. Descriptive statistics

Per feature: mean, sd, median, quartiles, IQR, range, coefficient of variation (CV), skewness, excess kurtosis and the
Shapiro-Wilk normality test.

In [6]:
desc = descriptive_statistics(table, info)
desc.to_csv(STAT_DIR / "descriptive_statistics.csv", index=False)
desc.head()

,feature,filter,image_type,feature_class,feature_name,n,mean,sd,median,q1,q3,iqr,min,max,cv,skewness,kurtosis,shapiro_p,normal
0,original_shape_Elongation,raw_image,original,shape,Elongation,40,0.783104,0.086565,0.770428,0.717577,0.812417,0.094840,0.683072,0.943640,0.110541,0.752388,-0.659714,0.000152,False
1,original_shape_Flatness,raw_image,original,shape,Flatness,40,0.569250,0.119254,0.566188,0.485570,0.656412,0.170843,0.404095,0.759277,0.209493,0.136986,-1.116946,0.007382,False
2,original_shape_LeastAxisLength,raw_image,original,shape,LeastAxisLength,40,62.604990,10.367206,62.862738,54.120326,71.051120,16.930795,48.263987,78.001302,0.165597,0.060356,-1.240895,0.003971,False
3,original_shape_MajorAxisLength,raw_image,original,shape,MajorAxisLength,40,111.418056,11.044410,112.129582,99.935128,119.437133,19.502006,99.248356,131.112977,0.099126,0.258143,-1.314253,0.000037,False
4,original_shape_Maximum2DDiameterColumn,raw_image,original,shape,Maximum2DDiameterColumn,40,122.680172,10.990695,127.949180,120.104121,130.080744,9.976623,96.254870,132.321578,0.089588,-1.409913,0.836794,0.000001,False


Normality and spread per filter (% normal = share of features with Shapiro-Wilk p ≥ 0.05)

In [7]:
normality_summary(desc, by="filter")

,features,normal_percent,median_abs_skewness,median_cv
filter,,,,
raw_image,78,30.8,0.608283,0.340519
square,71,35.2,0.489548,0.389021
square_root,62,35.5,0.665361,0.458029
logarithmic,61,26.2,0.915276,0.452384
exponential,68,35.3,0.508619,0.436566
gradient,66,45.5,0.472055,0.232290
log,370,31.4,0.435493,0.244106
wavelet,589,50.9,0.507794,0.285170
laws,433,37.6,0.535922,0.303889


Normality and spread per feature class

In [8]:
normality_summary(desc, by="feature_class")

,features,normal_percent,median_abs_skewness,median_cv
feature_class,,,,
shape,13,0.0,0.410798,0.168894
firstorder,448,49.6,0.430799,0.248978
glcm,525,57.3,0.424594,0.184414
glrlm,360,38.3,0.505602,0.259365
glszm,387,38.2,0.620882,0.272460
gldm,371,25.1,0.616715,0.400403
ngtdm,150,30.0,0.481712,0.453881


Descriptive statistics of every feature per MRI channel

In [9]:
desc_channel = descriptive_by_group(table, info["feature"], group="channel")
desc_channel.to_csv(STAT_DIR / "descriptive_statistics_by_channel.csv", index=False)
desc_channel.head(8)

,feature,channel,n,mean,sd,median,iqr
0,exponential_firstorder_10Percentile,0,10,14.178277,17.687647,6.327498,13.604234
1,exponential_firstorder_10Percentile,1,10,11.488780,8.105920,9.862468,11.039646
2,exponential_firstorder_10Percentile,2,10,14.194170,10.731956,12.863690,10.585385
3,exponential_firstorder_10Percentile,3,10,14.543910,12.486020,10.156011,11.936825
4,exponential_firstorder_90Percentile,0,10,128.774904,11.546287,128.203768,13.489201
5,exponential_firstorder_90Percentile,1,10,148.371071,11.268660,148.830506,11.037169
6,exponential_firstorder_90Percentile,2,10,157.351961,8.260575,156.397334,10.979980
7,exponential_firstorder_90Percentile,3,10,147.961601,14.416005,148.487320,19.400592


## 3. Univariate analysis

### 3a. Differences between MRI channels

Friedman test per feature; the 4 channels of the same breast form one block. Shape features are identical in every
channel (same mask), so they are not tested (p = NaN).

In [10]:
channel_results = repeated_measures_test(table, info, group="channel", block=["case", "side"], alpha=ALPHA)
channel_results.to_csv(STAT_DIR / "univariate_channel.csv", index=False)
channel_results.head(10)

channel: 253 of 2241 tested features differ (q < 0.05)


,feature,filter,image_type,feature_class,feature_name,n_blocks,statistic,p,effect,median_0,median_1,median_2,median_3,q,significant,test,effect_size
0,wavelet-HLH_firstorder_Maximum,wavelet,wavelet-HLH,firstorder,Maximum,10,24.24,0.000022,0.808,2.213374e+02,1.552301e+02,1.635133e+02,1.550898e+02,0.010365,True,Friedman,Kendall W
1,laws-L5S5S5-energy_firstorder_10Percentile,laws,laws-L5S5S5-energy,firstorder,10Percentile,10,22.68,0.000047,0.756,4.049615e+01,3.173092e+01,3.215792e+01,3.320596e+01,0.010365,True,Friedman,Kendall W
2,laws-E5E5E5-energy_firstorder_10Percentile,laws,laws-E5E5E5-energy,firstorder,10Percentile,10,22.44,0.000053,0.748,4.803973e+01,3.933545e+01,4.104390e+01,4.163771e+01,0.010365,True,Friedman,Kendall W
3,laws-R5R5R5-energy_firstorder_10Percentile,laws,laws-R5R5R5-energy,firstorder,10Percentile,10,22.44,0.000053,0.748,6.278227e+00,2.587544e+00,3.178535e+00,3.486433e+00,0.010365,True,Friedman,Kendall W
4,wavelet-HHL_firstorder_Variance,wavelet,wavelet-HHL,firstorder,Variance,10,22.32,0.000056,0.744,8.913369e+02,5.036010e+02,4.415220e+02,5.934207e+02,0.010365,True,Friedman,Kendall W
5,wavelet-HHL_firstorder_90Percentile,wavelet,wavelet-HHL,firstorder,90Percentile,10,21.72,0.000075,0.724,3.563274e+01,2.613490e+01,2.589203e+01,2.862478e+01,0.010365,True,Friedman,Kendall W
6,wavelet-HHL_firstorder_MeanAbsoluteDeviation,wavelet,wavelet-HHL,firstorder,MeanAbsoluteDeviation,10,21.72,0.000075,0.724,2.126490e+01,1.488692e+01,1.372942e+01,1.661344e+01,0.010365,True,Friedman,Kendall W
7,laws-R5R5R5-energy_firstorder_TotalEnergy,laws,laws-R5R5R5-energy,firstorder,TotalEnergy,10,21.24,0.000094,0.708,1.263443e+10,1.223347e+10,1.232875e+10,1.235214e+10,0.010365,True,Friedman,Kendall W
8,gabor-sigma-4-0-mm-lambda-4-0-mm-axial_firstor...,gabor,gabor-sigma-4-0-mm-lambda-4-0-mm-axial,firstorder,90Percentile,10,21.00,0.000105,0.700,3.825985e+00,3.287586e+00,3.388873e+00,3.662721e+00,0.010365,True,Friedman,Kendall W
9,laws-E5W5R5-energy_firstorder_10Percentile,laws,laws-E5W5R5-energy,firstorder,10Percentile,10,21.00,0.000105,0.700,3.000827e+01,1.793515e+01,1.984321e+01,2.106432e+01,0.010365,True,Friedman,Kendall W


Significant features per filter

In [11]:
significance_summary(channel_results, by="filter")

,tested,significant,median_effect,significant_percent
filter,,,,
raw_image,65,2,0.168,3.1
square,71,3,0.172,4.2
square_root,62,3,0.206,4.8
logarithmic,61,6,0.228,9.8
exponential,68,3,0.184,4.4
gradient,66,15,0.328,22.7
log,370,2,0.154,0.5
wavelet,589,77,0.208,13.1
laws,433,44,0.148,10.2


Significant features per feature class

In [12]:
significance_summary(channel_results, by="feature_class")

,tested,significant,median_effect,significant_percent
feature_class,,,,
firstorder,448,133,0.224,29.7
glcm,525,51,0.188,9.7
glrlm,360,18,0.172,5.0
glszm,387,16,0.172,4.1
gldm,371,17,0.188,4.6
ngtdm,150,18,0.204,12.0


Post-hoc test of the most significant feature: which channels differ? (pairwise Wilcoxon, Holm corrected)

In [13]:
top_feature = channel_results["feature"].iloc[0]
pairwise_posthoc(table, top_feature, group="channel", block=["case", "side"])

,feature,level_1,level_2,median_difference,p,p_holm
0,wavelet-HLH_firstorder_Maximum,0,1,56.113119,0.001953,0.011719
1,wavelet-HLH_firstorder_Maximum,0,2,40.617067,0.001953,0.011719
2,wavelet-HLH_firstorder_Maximum,0,3,50.608949,0.001953,0.011719
3,wavelet-HLH_firstorder_Maximum,1,2,-14.394862,0.001953,0.011719
4,wavelet-HLH_firstorder_Maximum,1,3,-3.874932,0.375000,0.375000
5,wavelet-HLH_firstorder_Maximum,2,3,7.356121,0.160156,0.320312


### 3b. Differences between left and right breast

Wilcoxon signed-rank test per feature; L and R of the same patient and channel form one pair (20 pairs).
Shape features are the same in every channel, so they are tested separately on one channel (5 pairs).

**Caution:** the 20 pairs come from only 5 patients, so the 4 channel pairs of a patient are not fully independent.
With 5 patients the smallest possible p-value at the patient level is 0.0625, so read these results as
exploratory.

In [14]:
info_shape = info[info["feature_class"] == "shape"]
info_no_shape = info[info["feature_class"] != "shape"]

In [15]:
side_results = repeated_measures_test(table, info_no_shape, group="side", block=["case", "channel"], alpha=ALPHA)
side_results.to_csv(STAT_DIR / "univariate_side.csv", index=False)
side_results.head(10)

side: 307 of 2241 tested features differ (q < 0.05)


,feature,filter,image_type,feature_class,feature_name,n_blocks,statistic,p,effect,median_L,median_R,q,significant,test,effect_size
0,laws-E5W5R5-energy_gldm_DependenceNonUniformity,laws,laws-E5W5R5-energy,gldm,DependenceNonUniformity,20,0.0,0.000002,-1.0,7.288236e+03,7.924054e+03,0.000427,True,Wilcoxon signed-rank,rank-biserial r
1,log-sigma-3-0-mm-3D_ngtdm_Strength,log,log-sigma-3-0-mm-3D,ngtdm,Strength,20,0.0,0.000002,1.0,9.121023e-02,7.792012e-02,0.000427,True,Wilcoxon signed-rank,rank-biserial r
2,log-sigma-2-0-mm-3D_ngtdm_Strength,log,log-sigma-2-0-mm-3D,ngtdm,Strength,20,0.0,0.000002,1.0,6.242401e-02,5.246963e-02,0.000427,True,Wilcoxon signed-rank,rank-biserial r
3,log-sigma-5-0-mm-3D_glrlm_RunLengthNonUniformi...,log,log-sigma-5-0-mm-3D,glrlm,RunLengthNonUniformityNormalized,20,0.0,0.000002,1.0,6.283148e-01,5.975330e-01,0.000427,True,Wilcoxon signed-rank,rank-biserial r
4,logarithm_firstorder_TotalEnergy,logarithmic,logarithm,firstorder,TotalEnergy,20,0.0,0.000002,-1.0,4.856228e+10,5.009487e+10,0.000427,True,Wilcoxon signed-rank,rank-biserial r
5,laws-E5E5E5-energy_glszm_ZoneVariance,laws,laws-E5E5E5-energy,glszm,ZoneVariance,20,0.0,0.000002,-1.0,1.618136e+05,3.468338e+05,0.000427,True,Wilcoxon signed-rank,rank-biserial r
6,log-sigma-4-0-mm-3D_ngtdm_Busyness,log,log-sigma-4-0-mm-3D,ngtdm,Busyness,20,0.0,0.000002,-1.0,7.672011e+00,8.368749e+00,0.000427,True,Wilcoxon signed-rank,rank-biserial r
7,laws-E5W5R5-energy_glrlm_RunLengthNonUniformity,laws,laws-E5W5R5-energy,glrlm,RunLengthNonUniformity,20,0.0,0.000002,-1.0,3.191572e+04,4.089979e+04,0.000427,True,Wilcoxon signed-rank,rank-biserial r
8,squareroot_firstorder_TotalEnergy,square_root,squareroot,firstorder,TotalEnergy,20,0.0,0.000002,-1.0,4.067026e+10,4.390517e+10,0.000427,True,Wilcoxon signed-rank,rank-biserial r
9,laws-L5E5E5-energy_firstorder_Energy,laws,laws-L5E5E5-energy,firstorder,Energy,20,0.0,0.000002,-1.0,2.581675e+10,2.615937e+10,0.000427,True,Wilcoxon signed-rank,rank-biserial r


In [16]:
side_shape_results = repeated_measures_test(table[table["channel"] == 0], info_shape, group="side", block=["case"], alpha=ALPHA)
side_shape_results.to_csv(STAT_DIR / "univariate_side_shape.csv", index=False)
side_shape_results

side: 0 of 13 tested features differ (q < 0.05)


,feature,filter,image_type,feature_class,feature_name,n_blocks,statistic,p,effect,median_L,median_R,q,significant,test,effect_size
0,original_shape_Maximum2DDiameterColumn,raw_image,original,shape,Maximum2DDiameterColumn,5,1.0,0.1250,-0.866667,120.104121,129.313572,0.609375,False,Wilcoxon signed-rank,rank-biserial r
1,original_shape_MeshVolume,raw_image,original,shape,MeshVolume,5,1.0,0.1250,-0.866667,131391.791667,130098.416667,0.609375,False,Wilcoxon signed-rank,rank-biserial r
2,original_shape_Elongation,raw_image,original,shape,Elongation,5,2.0,0.1875,0.733333,0.777322,0.719244,0.609375,False,Wilcoxon signed-rank,rank-biserial r
3,original_shape_VoxelVolume,raw_image,original,shape,VoxelVolume,5,2.0,0.1875,-0.733333,133146.000000,131376.000000,0.609375,False,Wilcoxon signed-rank,rank-biserial r
4,original_shape_SurfaceArea,raw_image,original,shape,SurfaceArea,5,3.0,0.3125,-0.600000,98494.301126,109152.124373,0.812500,False,Wilcoxon signed-rank,rank-biserial r
5,original_shape_Flatness,raw_image,original,shape,Flatness,5,4.0,0.4375,0.466667,0.590822,0.506130,0.902778,False,Wilcoxon signed-rank,rank-biserial r
6,original_shape_LeastAxisLength,raw_image,original,shape,LeastAxisLength,5,5.0,0.6250,-0.333333,65.167079,60.558398,0.902778,False,Wilcoxon signed-rank,rank-biserial r
7,original_shape_Maximum2DDiameterSlice,raw_image,original,shape,Maximum2DDiameterSlice,5,5.0,0.6250,-0.333333,121.753850,123.369364,0.902778,False,Wilcoxon signed-rank,rank-biserial r
8,original_shape_SurfaceVolumeRatio,raw_image,original,shape,SurfaceVolumeRatio,5,5.0,0.6250,0.333333,0.864310,0.766981,0.902778,False,Wilcoxon signed-rank,rank-biserial r
9,original_shape_MajorAxisLength,raw_image,original,shape,MajorAxisLength,5,6.0,0.8125,-0.200000,101.322191,117.877127,1.000000,False,Wilcoxon signed-rank,rank-biserial r


In [17]:
significance_summary(side_results, by="filter")

,tested,significant,median_effect,significant_percent
filter,,,,
raw_image,65,6,0.390476,9.2
square,71,8,0.371429,11.3
square_root,62,4,0.190476,6.5
logarithmic,61,3,0.190476,4.9
exponential,68,6,0.257143,8.8
gradient,66,7,0.176190,10.6
log,370,57,0.295238,15.4
wavelet,589,66,0.238095,11.2
laws,433,101,0.409524,23.3


### 3c. Where does the variance of a feature come from?

Share of every feature's variance explained by patient, side (inside patient), channel and the rest (side × channel).
A useful radiomic feature for patient-level questions should have a large *patient* share.

In [18]:
variance = variance_components(table, info)
variance.to_csv(STAT_DIR / "variance_components.csv", index=False)
variance.groupby("filter")[["patient", "side", "channel", "residual"]].mean().loc[[f for f in FILTER_ORDER if f in set(variance["filter"])]].round(3)

,patient,side,channel,residual
filter,,,,
raw_image,0.684,0.126,0.033,0.157
square,0.638,0.109,0.052,0.201
square_root,0.675,0.085,0.066,0.173
logarithmic,0.597,0.091,0.099,0.212
exponential,0.613,0.113,0.060,0.214
gradient,0.669,0.083,0.065,0.183
log,0.756,0.106,0.026,0.112
wavelet,0.584,0.117,0.075,0.224
laws,0.644,0.161,0.045,0.150


In [19]:
variance.groupby("feature_class")[["patient", "side", "channel", "residual"]].mean().round(3)

,patient,side,channel,residual
feature_class,,,,
firstorder,0.618,0.116,0.103,0.163
glcm,0.593,0.138,0.059,0.211
gldm,0.709,0.093,0.039,0.159
glrlm,0.671,0.103,0.042,0.185
glszm,0.646,0.126,0.036,0.191
ngtdm,0.753,0.126,0.029,0.093
shape,0.778,0.222,0.000,0.000


### 3d. Independent groups (for a clinical label)

There is no outcome label in the feature tables yet. When you add one (one value per patient, e.g. benign / malignant),
use one row per patient and channel, for example:

```python
labels = pd.read_csv(".../labels.csv")                     # columns: case, label
data = table[table["channel"] == 0].merge(labels, on="case")
outcome_results = independent_groups_test(data, info, group="label", alpha=ALPHA)
```

## 4. Relation among feature classes

Mean |Spearman rho| between all features of two classes. The diagonal shows how similar features within one class
still are after the redundancy step, the off-diagonal how strongly two classes are associated.

In [20]:
class_rho = mean_abs_spearman(table, info, by="feature_class")
class_rho.to_csv(STAT_DIR / "class_association_mean_abs_rho.csv")
class_rho.round(2)

,shape,firstorder,glcm,glrlm,glszm,gldm,ngtdm
shape,0.43,0.32,0.31,0.37,0.37,0.40,0.42
firstorder,0.32,0.35,0.33,0.35,0.34,0.37,0.35
glcm,0.31,0.33,0.34,0.35,0.33,0.38,0.35
glrlm,0.37,0.35,0.35,0.42,0.40,0.46,0.47
glszm,0.37,0.34,0.33,0.40,0.39,0.44,0.45
gldm,0.40,0.37,0.38,0.46,0.44,0.51,0.51
ngtdm,0.42,0.35,0.35,0.47,0.45,0.51,0.59


Modified RV coefficient between feature classes: a multivariate correlation (0-1) between two *blocks* of features.
It tells if two classes order the 40 samples in the same way overall.

In [21]:
class_rv = rv_matrix(table, info, by="feature_class")
class_rv.to_csv(STAT_DIR / "class_association_rv.csv")
class_rv.round(2)

,shape,firstorder,glcm,glrlm,glszm,gldm,ngtdm
shape,1.00,0.62,0.57,0.70,0.76,0.68,0.71
firstorder,0.62,1.00,0.86,0.82,0.82,0.80,0.63
glcm,0.57,0.86,1.00,0.85,0.79,0.84,0.57
glrlm,0.70,0.82,0.85,1.00,0.96,0.98,0.87
glszm,0.76,0.82,0.79,0.96,1.00,0.93,0.86
gldm,0.68,0.80,0.84,0.98,0.93,1.00,0.86
ngtdm,0.71,0.63,0.57,0.87,0.86,0.86,1.00


## 5. Relation among filters

Mean |Spearman rho| between the features of two filters

In [22]:
filter_rho = mean_abs_spearman(table, info, by="filter")
filter_rho.to_csv(STAT_DIR / "filter_association_mean_abs_rho.csv")
filter_rho.round(2)

,raw_image,square,square_root,logarithmic,exponential,gradient,log,wavelet,laws,gabor,binary_patterns
raw_image,0.44,0.45,0.44,0.41,0.40,0.38,0.39,0.35,0.31,0.37,0.39
square,0.45,0.46,0.44,0.41,0.42,0.40,0.38,0.35,0.30,0.35,0.41
square_root,0.44,0.44,0.47,0.47,0.41,0.40,0.41,0.36,0.30,0.38,0.39
logarithmic,0.41,0.41,0.47,0.47,0.39,0.38,0.40,0.35,0.29,0.36,0.36
exponential,0.40,0.42,0.41,0.39,0.42,0.41,0.37,0.35,0.29,0.33,0.40
gradient,0.38,0.40,0.40,0.38,0.41,0.52,0.46,0.44,0.37,0.44,0.44
log,0.39,0.38,0.41,0.40,0.37,0.46,0.47,0.40,0.36,0.43,0.41
wavelet,0.35,0.35,0.36,0.35,0.35,0.44,0.40,0.39,0.34,0.40,0.39
laws,0.31,0.30,0.30,0.29,0.29,0.37,0.36,0.34,0.39,0.36,0.33
gabor,0.37,0.35,0.38,0.36,0.33,0.44,0.43,0.40,0.36,0.43,0.37


Modified RV coefficient between filters (high = the two filters carry largely the same information)

In [23]:
filter_rv = rv_matrix(table, info, by="filter")
filter_rv.to_csv(STAT_DIR / "filter_association_rv.csv")
filter_rv.round(2)

,raw_image,square,square_root,logarithmic,exponential,gradient,log,wavelet,laws,gabor,binary_patterns
raw_image,1.00,0.94,0.91,0.79,0.84,0.61,0.69,0.66,0.53,0.64,0.68
square,0.94,1.00,0.87,0.74,0.92,0.66,0.65,0.65,0.49,0.58,0.72
square_root,0.91,0.87,1.00,0.95,0.83,0.60,0.70,0.63,0.47,0.60,0.61
logarithmic,0.79,0.74,0.95,1.00,0.71,0.52,0.65,0.56,0.43,0.55,0.50
exponential,0.84,0.92,0.83,0.71,1.00,0.72,0.68,0.69,0.48,0.59,0.75
gradient,0.61,0.66,0.60,0.52,0.72,1.00,0.82,0.91,0.68,0.82,0.79
log,0.69,0.65,0.70,0.65,0.68,0.82,1.00,0.85,0.73,0.88,0.76
wavelet,0.66,0.65,0.63,0.56,0.69,0.91,0.85,1.00,0.74,0.90,0.77
laws,0.53,0.49,0.47,0.43,0.48,0.68,0.73,0.74,1.00,0.77,0.60
gabor,0.64,0.58,0.60,0.55,0.59,0.82,0.88,0.90,0.77,1.00,0.68


The most strongly correlated feature pairs from *different* filters

In [24]:
cross_pairs = top_cross_pairs(table, info, by="filter", n=20)
cross_pairs.to_csv(STAT_DIR / "top_cross_filter_pairs.csv", index=False)
cross_pairs

,feature_1,filter_1,feature_2,filter_2,rho
0,squareroot_firstorder_RobustMeanAbsoluteDeviation,square_root,logarithm_firstorder_RobustMeanAbsoluteDeviation,logarithmic,0.999250
1,square_firstorder_10Percentile,square,exponential_firstorder_10Percentile,exponential,0.999062
2,gradient_glrlm_RunLengthNonUniformity,gradient,wavelet-HLL_glrlm_RunLengthNonUniformity,wavelet,0.997936
3,original_firstorder_10Percentile,raw_image,square_firstorder_10Percentile,square,0.997749
4,original_firstorder_Minimum,raw_image,exponential_firstorder_Minimum,exponential,0.997561
5,original_firstorder_TotalEnergy,raw_image,wavelet-LLL_firstorder_TotalEnergy,wavelet,0.997186
6,square_firstorder_Energy,square,wavelet-LLL_firstorder_TotalEnergy,wavelet,0.996998
7,original_firstorder_10Percentile,raw_image,squareroot_firstorder_10Percentile,square_root,0.996998
8,original_firstorder_10Percentile,raw_image,exponential_firstorder_10Percentile,exponential,0.996248
9,original_firstorder_InterquartileRange,raw_image,squareroot_firstorder_InterquartileRange,square_root,0.996060


## 6. Multivariate analysis

### 6a. Principal component analysis (PCA)

PCA of all standardized features. Explained variance per component:

In [25]:
pca_scores, pca_explained, pca_loadings = run_pca(table, info, n_components=10)
pca_scores.to_csv(STAT_DIR / "pca_scores.csv", index=False)
pca_loadings.to_csv(STAT_DIR / "pca_loadings.csv", index=False)
pca_explained.round(1)

,component,explained_percent,cumulative_percent
0,PC1,36.7,36.7
1,PC2,16.2,52.9
2,PC3,11.4,64.3
3,PC4,11.1,75.4
4,PC5,5.1,80.5
5,PC6,3.4,83.9
6,PC7,2.0,85.9
7,PC8,1.7,87.6
8,PC9,1.4,89.0
9,PC10,1.2,90.2


Features with the largest |loading| on PC1

In [26]:
pca_loadings.reindex(pca_loadings["PC1"].abs().sort_values(ascending=False).index).head(15)[["feature", "filter", "feature_class", "PC1", "PC2"]]

,feature,filter,feature_class,PC1,PC2
448,log-sigma-1-0-mm-3D_glrlm_RunLengthNonUniformi...,log,glrlm,0.033873,0.002437
536,log-sigma-2-0-mm-3D_gldm_DependenceNonUniformi...,log,gldm,0.033770,0.002971
2249,lbp-3D-k_gldm_SmallDependenceLowGrayLevelEmphasis,binary_patterns,gldm,0.033489,-0.005030
465,log-sigma-1-0-mm-3D_gldm_DependenceNonUniformi...,log,gldm,0.033422,-0.003315
473,log-sigma-1-0-mm-3D_gldm_SmallDependenceEmphasis,log,gldm,0.033354,0.003243
392,gradient_gldm_DependenceNonUniformityNormalized,gradient,gldm,0.033330,-0.004404
521,log-sigma-2-0-mm-3D_glrlm_RunLengthNonUniformi...,log,glrlm,0.033211,0.007533
2201,lbp-3D-m2_gldm_SmallDependenceEmphasis,binary_patterns,gldm,0.033168,-0.004503
1115,wavelet-HLH_glrlm_RunLengthNonUniformityNormal...,wavelet,glrlm,0.033166,-0.000048
817,wavelet-LLH_glrlm_RunLengthNonUniformityNormal...,wavelet,glrlm,0.033148,0.003119


### 6b. PERMANOVA

Do the multivariate centroids differ between groups? R² = share of the total variance explained by the grouping.
Labels are permuted only inside the blocks, so the repeated-measures design is kept.

In [27]:
permanova_channel = permanova(table, info, group="channel", strata=["case", "side"], n_perm=N_PERMUTATIONS)
permanova_channel

group                channel
strata          [case, side]
pseudo_F            0.703159
R2                  0.055353
p                      0.002
permutations             999
dtype: object

In [28]:
permanova_side = permanova(table, info_no_shape, group="side", strata=["case", "channel"], n_perm=N_PERMUTATIONS)
permanova_side

group                      side
strata          [case, channel]
pseudo_F               0.850372
R2                     0.021888
p                         0.002
permutations                999
dtype: object

In [29]:
permanova_patient = permanova(table, info, group="case", strata=["channel"], n_perm=N_PERMUTATIONS)
permanova_patient

group                case
strata          [channel]
pseudo_F         16.26474
R2               0.650206
p                   0.001
permutations          999
dtype: object

In [30]:
permanova_results = pd.DataFrame([permanova_channel, permanova_side, permanova_patient])
permanova_results.to_csv(STAT_DIR / "permanova.csv", index=False)
permanova_results

,group,strata,pseudo_F,R2,p,permutations
0,channel,"[case, side]",0.703159,0.055353,0.002,999
1,side,"[case, channel]",0.850372,0.021888,0.002,999
2,case,[channel],16.264740,0.650206,0.001,999


Dispersion check: PERMANOVA can also be significant when groups only differ in spread. If this test is *not*
significant, a significant PERMANOVA reflects a difference in location (centroid).

In [31]:
dispersion_test(table, info, group="channel")

group                                                         channel
mean_distance_to_centroid    {0: 44.58, 1: 45.02, 2: 47.77, 3: 42.55}
H                                                            1.356585
p                                                            0.715742
dtype: object

### 6c. Cross-validated classification

Can all features together predict the channel / side of a sample? Logistic regression with leave-one-patient-out
cross-validation; the p-value comes from repeating the whole procedure with permuted labels.

In [32]:
channel_cv, channel_null = classification_cv(table, info, target="channel", cv_groups="case", n_perm=200)
channel_cv

target                channel
balanced_accuracy       0.375
chance_level             0.25
permutation_mean         0.25
p                    0.034826
permutations              200
dtype: object

In [33]:
side_cv, side_null = classification_cv(table, info_no_shape, target="side", cv_groups="case", n_perm=200)
side_cv

target                   side
balanced_accuracy       0.525
chance_level              0.5
permutation_mean      0.49925
p                    0.442786
permutations              200
dtype: object

In [34]:
cv_results = pd.DataFrame([channel_cv, side_cv])
cv_results.to_csv(STAT_DIR / "classification_cv.csv", index=False)
cv_results

,target,balanced_accuracy,chance_level,permutation_mean,p,permutations
0,channel,0.375,0.25,0.25000,0.034826,200
1,side,0.525,0.50,0.49925,0.442786,200


## 7. How to read the results

- **Patient vs. channel/side:** if the *patient* share of the variance (3c) and the PERMANOVA R² for `case` are much
  larger than those for `channel` and `side`, the features mainly describe the patient (i.e. the breast tissue),
  which is what a radiomic signature should do.
- **Channel effect:** features with a large Kendall's W are channel-dependent. Analyse them per channel, or harmonize
  them (next step in the pipeline), before you pool channels.
- **Filters:** a high RV / mean |rho| between two filters means they add little new information; a low value means
  a filter captures something different (e.g. Laws, Gabor and LBP texture vs. the intensity filters).
- **Sample size:** 5 patients is small. Treat p-values as exploratory and confirm on more patients.